# Validação do Pipeline
Este notebook executa e valida cada etapa do pipeline de processamento de dados e geração de resultados. Cada etapa é executada separadamente para facilitar o acompanhamento e depuração.

## Importação de Bibliotecas Necessárias
Importamos as bibliotecas essenciais para execução de scripts e validação de arquivos de saída.

In [19]:
# Importação de bibliotecas
import subprocess
import os
import sys
from variaveis import INPUT_PATHS, OUTPUT_PATHS, CARBONO_CONSOLIDADO

## Funções Auxiliares
Definimos funções auxiliares para impressão segura e verificação de arquivos de saída.

In [20]:
# Função para impressão segura
def safe_print(s: str):
    try:
        sys.stdout.write(s)
    except UnicodeEncodeError:
        clean = s.encode(sys.stdout.encoding, errors='replace').decode(
            sys.stdout.encoding)
        sys.stdout.write(clean)

# Função para verificar existência de arquivos
def check_files(files):
    status = True
    for f in files:
        if not os.path.exists(f):
            print(f"[MISSING] {f}")
            status = False
        else:
            print(f"[OK]      {f}")
    return status


# Initialize overall success tracker
all_ok = True

## Etapa 1: Executar `00_extrair_pib_municipal.py`
Executa o script responsável por extrair os dados de PIB municipal e valida os arquivos de saída.

In [ ]:
# Etapa 1
script = "codingCO2/00_extrair_pib_municipal.py"
outputs = [OUTPUT_PATHS.pib_ibge_csv]

display(f"\n=== Running {script} ===")
result = subprocess.run([sys.executable, script], capture_output=True, text=True)
display(result.stdout)
if result.returncode != 0:
    safe_print(f"[ERROR] {script} failed (exit code {result.returncode})\n")
    all_ok = False
else:
    display(f"Checking outputs for {script}:")
    if not check_files(outputs):
        all_ok = False

'\n=== Running codingCO2/00_extrair_pib_municipal.py ==='

''

[ERROR] codingCO2/00_extrair_pib_municipal.py failed (exit code 2)


## Etapa 2: Executar `01_extrair_gee_municipal_excel.py`
Executa o script responsável por extrair os dados de GEE municipal e valida os arquivos de saída.

In [ ]:
# Etapa 2
script = "codingCO2/01_extrair_cobertura_municipal.py"
outputs = [OUTPUT_PATHS.mapbiomas_long_csv]

display(f"=== Running {script} ===")
result = subprocess.run([sys.executable, script], capture_output=True, text=True)
display(result.stdout)
if result.returncode != 0:
    safe_print(f"[ERROR] {script} failed (exit code {result.returncode})\n")
    all_ok = False
else:
    display(f"Checking outputs for {script}:")
    if not check_files(outputs):
        all_ok = False

'=== Running 01_extrair_cobertura_municipal.py ==='

''

[ERROR] 01_extrair_cobertura_municipal.py failed (exit code 1)


## Etapa 3: Executar `02_extrair_alertas_desmatamento.py`
Executa o script responsável por extrair os alertas de desmatamento e valida os arquivos de saída.

In [ ]:
# Etapa 3
script = "codingCO2/02_extrair_alertas_desmatamento.py"
outputs = [OUTPUT_PATHS.alertas_csv]

display(f"=== Running {script} ===")
result = subprocess.run([sys.executable, script], capture_output=True, text=True)
display(result.stdout)
if result.returncode != 0:
    safe_print(f"[ERROR] {script} failed (exit code {result.returncode})\n")
    all_ok = False
else:
    display(f"Checking outputs for {script}:")
    if not check_files(outputs):
        all_ok = False

'=== Running 02_extrair_alertas_desmatamento.py ==='

''

[ERROR] 02_extrair_alertas_desmatamento.py failed (exit code 1)


## Etapa 4: Executar `03_extrair_uso_terra_timeseries.py`
Executa o script responsável por extrair as séries temporais de uso da terra e valida os arquivos de saída.

In [ ]:
# Etapa 4
script = "codingCO2/03_extrair_uso_terra_timeseries.py"
outputs = [INPUT_PATHS.uso_timeseries]

display(f"=== Running {script} ===")
result = subprocess.run([sys.executable, script], capture_output=True, text=True)
display(result.stdout)
if result.returncode != 0:
    safe_print(f"[ERROR] {script} failed (exit code {result.returncode})\n")
    all_ok = False
else:
    display(f"Checking outputs for {script}:")
    if not check_files(outputs):
        all_ok = False

'=== Running 03_extrair_uso_terra_timeseries.py ==='

''

[ERROR] 03_extrair_uso_terra_timeseries.py failed (exit code 1)


## Etapa 5: Executar `04_consolidar_modelar_carbono.py`
Executa o script responsável por consolidar e modelar os dados de carbono e valida os arquivos de saída.

In [ ]:
# Etapa 5
script = "codingCO2/04_consolidar_dados_carbono.py"
outputs = [
    CARBONO_CONSOLIDADO,
    OUTPUT_PATHS.model_results_csv
]

display(f"=== Running {script} ===")
result = subprocess.run([sys.executable, script], capture_output=True, text=True)
display(result.stdout)
if result.returncode != 0:
    safe_print(f"[ERROR] {script} failed (exit code {result.returncode})\n")
    all_ok = False
else:
    display(f"Checking outputs for {script}:")
    if not check_files(outputs):
        all_ok = False

'=== Running 04_consolidar_dados_carbono.py ==='

''

[ERROR] 04_consolidar_dados_carbono.py failed (exit code 1)


## Etapa 6: Executar `05_gerar_figuras_carbono.py`
Executa o script responsável por gerar as figuras de carbono e valida os arquivos de saída dinamicamente.

In [ ]:
# Etapa 6
script = "codingCO2/05_gerar_figuras_carbono.py"

display(f"=== Running {script} ===")
result = subprocess.run([sys.executable, script], capture_output=True, text=True)
display(result.stdout)
if result.returncode != 0:
    safe_print(f"[ERROR] {script} failed (exit code {result.returncode})\n")
    all_ok = False
else:
    display(f"Checking outputs for {script}:")
    fig_dir = os.path.dirname(OUTPUT_PATHS.evolucao_pib_png)
    expected = []
    for i in range(1, 6):
        expected.append(os.path.join(fig_dir, f"Figura{i:02d}_*.png"))
    for i in range(1, 10):
        expected.append(os.path.join(fig_dir, f"Figura07_{i}_*.png"))
    expected.append(os.path.join(fig_dir, "Figura08_Importancia_Variaveis.png"))
    expected.append(os.path.join(fig_dir, "Figura09_Evolucao_Preco_Carbono.png"))
    import glob
    for pattern in expected:
        matches = glob.glob(pattern)
        if matches:
            display(f"[OK]      Pattern {pattern} -> {len(matches)} file(s)")
        else:
            display(f"[MISSING] Pattern {pattern}")
        all_ok = all_ok and len(matches) > 0

'=== Running 05_gerar_figuras_carbono.py ==='

'[INFO] Carregando dados consolidados de: data/generated/carbono_serra_penitente.csv\n'

[ERROR] 05_gerar_figuras_carbono.py failed (exit code 1)


## Conclusão
Após a execução de todas as etapas, verificamos se todos os arquivos de saída esperados foram gerados corretamente.

In [27]:
# Conclusão
# all_ok = result.returncode == 0  # Check if the last step ran successfully
print("Pipeline validation completed.")
if not all_ok:
    print("Some steps failed or outputs are missing.")
    sys.exit(1)
else:
    print("All steps ran successfully and outputs are present.")

Pipeline validation completed.
Some steps failed or outputs are missing.


SystemExit: 1